# CodeGenTutor — fine-tuning a question generator

Teaches a small model to write a LeetCode-style problem **in the exact JSON schema
`data/questions/*.json` already uses**, so a generated question drops into the app's
sandbox, evaluator and recommender without any of them changing.

| | |
|---|---|
| **Base model** | `unsloth/Qwen2.5-Coder-3B-Instruct` (4-bit) |
| **Data** | `newfacade/LeetCodeDataset`, filtered by this repo's own ingest pipeline |
| **Method** | QLoRA (r=16) via Unsloth |
| **Output** | `codegen-tutor.Q4_K_M.gguf`, ~1.9–2.2 GB, runs in Ollama |
| **Runtime** | ~60–90 min end to end on a free T4 |

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**.

### What this notebook does that a plain fine-tuning notebook doesn't

It imports the repo's ingest and verification code instead of reimplementing it. That is
not tidiness — it is the only thing that guarantees the JSON this model is trained to emit
is the JSON `sandbox/runner.py` can execute. A notebook with its own copy of the schema
drifts from the app the first time either side is edited, and the symptom is a model that
looks fine here and produces unservable questions in the app.

The same idea drives the quality filter and the success metric: both are the repo's real
checkers, run on the model's real output.

In [ ]:
# Fail now, not 40 minutes in.
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
)

print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

In [ ]:
# The cell most likely to rot. If a rerun months from now breaks here, pin the
# versions that worked rather than debugging the dependency graph.
%pip install -q unsloth
%pip install -q --no-deps --upgrade "trl>=0.15" peft accelerate bitsandbytes
%pip install -q datasets

In [ ]:
# Clone the app and reuse its pipeline. THIS is what keeps training-time JSON and
# run-time JSON the same object.
#
# Safe to rerun: it re-clones from scratch and clears the two caches that make a
# re-clone look like a broken repo.
import importlib
import shutil
import sys
from pathlib import Path

REPO = "https://github.com/Burkifa23/4ALL.git"
BRANCH = "experimental-2"  # the branch carrying evaluator/generate.py

REPO_PATH = Path("/content/4ALL")

if REPO_PATH.exists():
    shutil.rmtree(REPO_PATH)

!git clone -q --branch {BRANCH} --depth 1 {REPO} {REPO_PATH}

# Check for the NEWEST file, not an old one. A clone that succeeds against a
# branch that predates the custom-practice work looks perfectly healthy right up
# until the import below fails.
assert (REPO_PATH / "evaluator" / "generate.py").exists(), (
    f"evaluator/generate.py is not on {BRANCH!r} at the remote. It is the module "
    "this notebook shares its prompt and schema with, so training cannot start "
    "without it: commit and push the custom-practice work, or upload the file "
    "into /content/4ALL/evaluator/ from the Colab Files pane."
)

# data/ ships without __init__.py and imports fine as a namespace package, but a
# namespace package merges every `data` directory on sys.path — and /content/data
# is a folder people really do create in Colab. Making it a regular package
# stops that.
for package in (REPO_PATH / "data", REPO_PATH / "data" / "ingest"):
    (package / "__init__.py").touch()

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# The fix for a "No module named 'data'" that survives an obviously-successful
# clone: the import system caches a listing of every directory on sys.path and
# only re-reads it when that directory's mtime changes. Deleting and re-cloning
# a tree that was already on sys.path is exactly the case that check misses, so
# Python keeps serving a listing from before these files existed. Anything
# created after the interpreter started needs this call.
importlib.invalidate_caches()

# Same hazard one level up: a rerun re-clones, but modules imported by the
# previous run stay live in sys.modules and would shadow the fresh checkout.
for name in [n for n in sys.modules if n.split(".")[0] in
             {"data", "evaluator", "contracts", "sandbox", "recommender"}]:
    del sys.modules[name]

# The ingest pipeline: HF rows -> the app's question schema.
from data.ingest.ingest_leetcode import filter_keep_pile, load_raw, to_question_record

# Execs a question's reference_solution against its own test_cases; [] means sound.
from data.ingest.verify_solutions import run_one

# The prompt, the instruction format and the six taught fields — shared verbatim
# with the app, so what we train is what CodeGenTutor is later asked for.
from evaluator import generate
from evaluator.errors import EvaluatorError
from evaluator.generate import (
    SOLUTION_PREAMBLE,
    SYSTEM_PROMPT,
    TAUGHT_FIELDS,
    _parse,
    instruction,
)
from evaluator.parsing import strip_fences

generate.GENERATED_DIR = Path("/content/generated")  # keep the clone clean

print("taught fields:", TAUGHT_FIELDS)
print()
print(SYSTEM_PROMPT)

## The data contract

A question file has 14 fields. **The model is taught six.**

| Taught | Why |
|---|---|
| `title`, `description_md` | the problem, as the student reads it |
| `starter_code`, `entry_point` | must agree with each other exactly — `sandbox/runner_worker.py` does `eval(entry_point)` and calls it |
| `reference_solution` | what makes the test cases checkable |
| `test_cases` | `[{"input": {param: value}, "expected": value}]` |

Everything else is filled in by `evaluator/generate.py::_finalize`:

- `difficulty` and `topic` are **inputs** — they're in the instruction. Teaching a model to
  echo its own prompt back spends tokens and adds a field that can disagree with the request.
- `question_id`, `topics`, `test_case_count`, `optimal_complexity`, `source_*` are bookkeeping.
- The 60-line import preamble is re-attached at runtime from `SOLUTION_PREAMBLE`, so the model
  never spends tokens on imports and can't forget one.

Two dataset facts drive the preparation below: rows carry **up to 93 test cases**, and every
reference solution repeats that preamble. Left alone, a single example runs past 20k tokens and
nothing fits on a T4.

In [ ]:
# ~5 min. filter_keep_pile applies the ingest rules: single-method entry points,
# parseable starter code, usable test data, stdlib only, no tree/linked-list
# problems (the flat test_cases schema can't express node reconstruction).
df = load_raw()
print(f"{len(df)} rows across all splits")

keep = filter_keep_pile(df)
print(f"{len(keep)} usable  ({len(df) - len(keep)} dropped by the ingest filters)")

In [ ]:
import random

random.seed(42)

MAX_TESTS = 8


def solution_body(record):
    """Just the `class Solution` block. The dataset's import preamble is
    re-attached at runtime from SOLUTION_PREAMBLE, so training on it would be
    paying ~500 tokens an example to teach the model something it is given."""
    source = record["reference_solution"]
    cut = source.rfind("class Solution:")
    return source[cut:] if cut != -1 else None


def trim(record):
    """Cap the test cases and drop the preamble. Keeps the first three (the
    dataset's own worked examples, which the description refers to) and samples
    the rest so edge cases aren't systematically lost to truncation."""
    body = solution_body(record)
    if body is None:
        return None

    cases = record["test_cases"]
    if len(cases) > MAX_TESTS:
        cases = cases[:3] + random.sample(cases[3:], MAX_TESTS - 3)

    return {**record, "reference_solution": body, "test_cases": cases,
            "test_case_count": len(cases)}


trimmed = []
for i, (_, row) in enumerate(keep.iterrows(), start=1):
    record = to_question_record(row, i)
    if not record["test_cases"] or len(record["test_cases"]) < 4:
        continue
    record = trim(record)
    if record is not None:
        trimmed.append(record)

print(f"{len(trimmed)} trimmed to <= {MAX_TESTS} test cases")

In [ ]:
# The quality gate: keep only questions whose reference solution really passes
# their own test cases, run under the SAME preamble the app uses at runtime.
#
# Two things get checked at once — that the dataset row is sound, and that
# SOLUTION_PREAMBLE covers what these solutions actually reference. A wave of
# NameErrors here means the preamble is missing an import, not that the data is bad.
#
# ~10 min. run_one() has no timeout, so a pathological row can stall the cell; if
# it does, note the index printed last and skip it.
good, dropped = [], []

for i, record in enumerate(trimmed):
    if i % 250 == 0:
        print(f"  {i}/{len(trimmed)} ...")

    checkable = {**record, "reference_solution": SOLUTION_PREAMBLE + record["reference_solution"]}

    try:
        failures = run_one(checkable)
    except Exception as exc:                      # a row that breaks the checker itself
        failures = [f"{type(exc).__name__}: {exc}"]

    (good if not failures else dropped).append((record, failures))

good = [record for record, _ in good]

print(f"\n{len(good)} verified  ({len(dropped)} dropped as inconsistent)")
print("\nsample of what was dropped and why:")
for record, failures in dropped[:5]:
    print(f"  {record['title'][:40]:42} {failures[0][:90]}")

In [ ]:
import json
from collections import Counter

print(Counter(r["difficulty"] for r in good), "1=Easy 2=Medium 3=Hard")
print(len(Counter(r["topic"] for r in good)), "distinct topics")


def target_json(record):
    """What the assistant turn must produce. Compact, not indented: indentation
    is ~20% more tokens per example to teach whitespace nobody reads."""
    return json.dumps({field: record[field] for field in TAUGHT_FIELDS}, ensure_ascii=False)


def pair(record):
    return {"conversations": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction(record["topic"], record["difficulty"])},
        {"role": "assistant", "content": target_json(record)},
    ]}


pairs = [pair(record) for record in good]

print()
print(pairs[0]["conversations"][1]["content"])
print(pairs[0]["conversations"][2]["content"][:400], "...")

In [ ]:
# Hold out whole TOPICS, not random rows.
#
# A random 5% split measures memorisation: the same topic in train and eval lets
# the model recall a problem it has seen. Held-out topics measure the thing that
# actually matters here — can it write valid JSON for a topic nobody trained it on,
# which is exactly what a student typing "Sliding Window" is asking for.
topics = sorted({record["topic"] for record in good})
random.Random(42).shuffle(topics)

held_out = set(topics[: max(2, len(topics) // 10)])

train_pairs = [p for p, r in zip(pairs, good) if r["topic"] not in held_out]
eval_records = [r for r in good if r["topic"] in held_out]

print(f"train {len(train_pairs)}   held-out topics {sorted(held_out)}")

## LoRA settings, and why

**4-bit base + LoRA adapters.** A 3B model in fp16 is ~6 GB of weights before optimiser state
and activations; a T4 has 16 GB. Quantising the frozen base to 4-bit and training ~30M adapter
parameters is what makes this fit at a 4096 context.

**`r=16`, `lora_alpha=16`.** This is a formatting task — the model already writes Python, and is
being taught a *shape*: which keys, in which order, with `entry_point` agreeing with
`starter_code`. That needs far less capacity than teaching new knowledge. Higher r mostly buys
overfitting to the 2k examples.

**`max_seq_length=4096`.** A description plus eight test cases lands around 1200–2500 tokens.
4096 leaves headroom without paying for attention over context nothing uses.

**Targeting all seven projections** (q, k, v, o, gate, up, down) rather than attention only:
standard for instruction tuning, and cheap at this rank.

**2 epochs.** Enough for schema compliance; more starts reproducing training problems verbatim,
which is the failure mode that makes a "custom" question generator worthless.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # let Unsloth pick for the GPU it finds
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
from datasets import Dataset


def render(example):
    """The model's own chat template — not a hand-written one. A template that
    disagrees with what Ollama applies at serving time is the classic way a
    fine-tune that trained perfectly produces garbage in the app."""
    return {"text": tokenizer.apply_chat_template(example["conversations"], tokenize=False)}


train_ds = Dataset.from_list(train_pairs).map(render)

lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"]]
print(f"tokens per example: median {sorted(lengths)[len(lengths) // 2]}, max {max(lengths)}")

over = sum(1 for n in lengths if n > MAX_SEQ_LENGTH)
print(f"{over} examples over {MAX_SEQ_LENGTH} tokens - dropping them (a truncated "
      f"example teaches the model to emit unterminated JSON)")

train_ds = train_ds.select([i for i, n in enumerate(lengths) if n <= MAX_SEQ_LENGTH])

print(f"\ntraining on {len(train_ds)} examples\n")
print(train_ds[0]["text"][:1200])

In [ ]:
# The scorecard. Three numbers, measured on held-out topics, using the app's own
# checkers — _parse() is literally what the app calls on CodeGenTutor's reply, and
# run_one() is the check that decides whether a student gets a solvable question.
#
# Run BEFORE training: the adapters are zero-initialised, so this is the base model.
import time

N_EVAL = 20

eval_requests = [(r["topic"], r["difficulty"]) for r in eval_records][:N_EVAL]


def generate_one(topic, difficulty, max_new_tokens=1600):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": instruction(topic, difficulty)}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.6, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def score(label):
    FastLanguageModel.for_inference(model)
    counts = {"valid JSON": 0, "all 6 fields": 0, "tests self-consistent": 0}
    started = time.time()

    for topic, difficulty in eval_requests:
        text = generate_one(topic, difficulty)

        try:
            json.loads(strip_fences(text))
        except Exception:
            continue
        counts["valid JSON"] += 1

        try:
            record = _parse(text, topic, difficulty)   # the app's own acceptance check
        except EvaluatorError:
            continue
        counts["all 6 fields"] += 1

        try:
            if not run_one(record):
                counts["tests self-consistent"] += 1
        except Exception:
            pass

    n = len(eval_requests)
    print(f"{label}  ({time.time() - started:.0f}s)")
    for name, hits in counts.items():
        print(f"   {name:24} {hits}/{n}  ({hits / n:.0%})")
    return counts


before = score("BEFORE fine-tuning (base model)")

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch 8; 2 is what fits a T4
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# Loss on the JSON only. Without this the model also spends capacity learning to
# predict the instruction — which it is always given, never asked to write.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Older TRL: SFTConfig doesn't exist; pass transformers.TrainingArguments instead
# and move dataset_text_field / max_seq_length onto SFTTrainer itself.

In [ ]:
# ~30-60 min on a T4. Watch the loss: a steady fall to ~0.3-0.5 is healthy;
# flat near zero within a few hundred steps means it is memorising, not learning
# the schema — cut to 1 epoch and rerun.
stats = trainer.train()

used = torch.cuda.max_memory_reserved() / 1e9
print(f"\n{stats.metrics['train_runtime'] / 60:.1f} min, peak VRAM {used:.1f} GB")

In [ ]:
after = score("AFTER fine-tuning")

print("\n" + "=" * 58)
print(f"{'metric':26} {'before':>10} {'after':>10}   {'delta':>8}")
print("=" * 58)
for name in before:
    b, a = before[name], after[name]
    print(f"{name:26} {b:>7}/{N_EVAL} {a:>7}/{N_EVAL}   {a - b:>+8}")
print("=" * 58)
print("\nHeld-out topics, so this is generalisation, not recall.")
print("If the deltas are ~0, say so in the report — a fine-tune that changed")
print("nothing is a finding, and a few-shot prompt on the base model would then")
print("be the cheaper answer.")

In [ ]:
# Eyeball one. The metrics say the JSON is well-formed and self-consistent; they
# cannot say whether the description matches the tests, and only a human can.
topic, difficulty = eval_requests[0]

text = generate_one(topic, difficulty)
record = _parse(text, topic, difficulty)

print(f"{record['title']}   [{topic}, difficulty {difficulty}]\n")
print(record["description_md"][:900])
print("\n--- starter code ---")
print(record["starter_code"])
print("--- entry point ---", record["entry_point"])
print("--- test cases ---")
for case in record["test_cases"][:4]:
    print("   ", case)
print("\nreference solution passes its own tests:", not run_one(record))

In [ ]:
# ~15 min: builds llama.cpp, merges the adapters, quantises. The flakiest cell in
# the notebook — it needs several GB of free disk, and Colab's llama.cpp build
# breaks from time to time. If it fails, use the fallback in the next cell.
model.save_pretrained_gguf(
    "codegen-tutor",
    tokenizer,
    quantization_method="q4_k_m",   # ~1.9-2.2 GB for a 3B; the size/quality knee
)

!ls -lh codegen-tutor/*.gguf

In [ ]:
# Fallback, only if the cell above failed: push merged fp16 weights and convert
# on your own machine with llama.cpp's convert_hf_to_gguf.py.
#
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub_merged("<your-hf-user>/codegen-tutor", tokenizer,
#                          save_method="merged_16bit")

# Getting the 2 GB file out. A browser download of that size drops often enough
# that the Hub is the better default; files.download is the no-account path.
#
# model.push_to_hub_gguf("<your-hf-user>/codegen-tutor-gguf", tokenizer,
#                        quantization_method="q4_k_m", token="hf_...")

from google.colab import files

files.download("codegen-tutor/unsloth.Q4_K_M.gguf")

## Handoff to Ollama

1. Put the downloaded file in the repo as `models/codegen-tutor.Q4_K_M.gguf` (the name
   `models/Modelfile.codegen-tutor` expects — rename it, Unsloth exports as
   `unsloth.Q4_K_M.gguf`).

2. Build the model:

   ```bash
   ollama create CodeGenTutor -f models/Modelfile.codegen-tutor
   ```

3. In the app's sidebar: **Settings** → Local (Ollama), then **Custom Practice** → type a
   topic, pick a difficulty, leave the generator model as `CodeGenTutor`.

`models/Modelfile.codegen-tutor` carries the same system prompt this notebook trained
against, and `tests/test_generated_question.py` asserts the two stay byte-identical — a model
served under a different system prompt than it was tuned on is the most common way a working
fine-tune looks broken.

### If the generated questions are rejected in the app

`evaluator/generate.py` runs every generated question through the real sandbox before serving
it, and retries once. Persistent rejection with a healthy scorecard above usually means the
Modelfile, not the model: check `num_ctx 4096` (a truncated reply is never valid JSON) and
that the `SYSTEM` block still matches.